# Persistent Homology of Fibonacci Cubes as Metric Spaces

This notebook is analogous to the Möbius ladder and knight graph notebooks, but for **Fibonacci cubes**.

The Fibonacci cube \(\Gamma_n\) is the induced subgraph of the \(n\)-dimensional hypercube \(Q_n\) on the binary strings of length \(n\) with **no two consecutive 1s**. Two vertices are adjacent exactly when the corresponding binary strings differ in one coordinate.

We view \(\Gamma_n\) as a finite metric space using the unweighted graph shortest-path metric, then compute Vietoris--Rips persistent homology using `ripser(..., distance_matrix=True)`.

Main tasks:

1. Build Fibonacci cubes \(\Gamma_n\).
2. Compute shortest-path distance matrices.
3. Compute and plot persistence diagrams and barcodes.
4. Run ordinary batch experiments through dimension 3.
5. Run a large-scale positive-dimensional barcode export for later pattern analysis.

The convention here includes \(\Gamma_0\) as a single vertex, but most persistence experiments start at \(n=1\) or higher.

In [ ]:
# If needed, uncomment and run this cell once.
# %pip install ripser persim numpy scipy pandas matplotlib ipywidgets networkx

In [ ]:
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

from ripser import ripser
from persim import plot_diagrams

try:
    import networkx as nx
    HAS_NETWORKX = True
except Exception:
    HAS_NETWORKX = False

plt.rcParams["figure.figsize"] = (7, 5)

## 1. Construct Fibonacci cubes and graph metrics

A vertex is a length-`n` bitstring with no adjacent 1s. Edges connect strings that differ in exactly one bit.

In [ ]:
def fibonacci_number(k: int):
    """Return Fibonacci number F_k with F_0=0, F_1=1."""
    if k < 0:
        raise ValueError("k must be nonnegative")
    a, b = 0, 1
    for _ in range(k):
        a, b = b, a + b
    return a


def fibonacci_strings(n: int):
    """Return all binary strings of length n with no consecutive 1s.

    Strings are returned in lexicographic order as tuples of 0/1 integers.
    The number of such strings is F_{n+2}.
    """
    if n < 0:
        raise ValueError("n must be nonnegative")

    out = []

    def rec(prefix, last_bit, remaining):
        if remaining == 0:
            out.append(tuple(prefix))
            return
        # add 0
        prefix.append(0)
        rec(prefix, 0, remaining - 1)
        prefix.pop()
        # add 1 if previous bit was not 1
        if last_bit != 1:
            prefix.append(1)
            rec(prefix, 1, remaining - 1)
            prefix.pop()

    rec([], 0, n)
    return out


def bitstring_to_str(bits):
    return "".join(str(b) for b in bits)


def fibonacci_cube_edges(n: int):
    """Return undirected edges of the Fibonacci cube Gamma_n.

    Vertices are indexed by the order of fibonacci_strings(n).
    """
    vertices = fibonacci_strings(n)
    index = {v: i for i, v in enumerate(vertices)}
    edges = set()

    for i, bits in enumerate(vertices):
        for j in range(n):
            flipped = list(bits)
            flipped[j] = 1 - flipped[j]
            flipped = tuple(flipped)
            if flipped in index:
                k = index[flipped]
                if i != k:
                    edges.add(tuple(sorted((i, k))))
    return sorted(edges), vertices


def fibonacci_cube_adjacency(n: int):
    """Sparse adjacency matrix for Gamma_n."""
    edges, vertices = fibonacci_cube_edges(n)
    m = len(vertices)
    if not edges:
        return csr_matrix((m, m), dtype=float), vertices

    rows, cols = [], []
    for u, v in edges:
        rows.extend([u, v])
        cols.extend([v, u])
    data = np.ones(len(rows), dtype=float)
    return csr_matrix((data, (rows, cols)), shape=(m, m)), vertices


def fibonacci_cube_distance_matrix(n: int):
    """All-pairs unweighted shortest-path distance matrix for Gamma_n."""
    A, vertices = fibonacci_cube_adjacency(n)
    D = shortest_path(A, directed=False, unweighted=True)
    D = np.asarray(D, dtype=float)
    if not np.all(np.isfinite(D)):
        raise ValueError(f"Gamma_{n} appears disconnected; distance matrix has infinities")
    return D, vertices


def graph_diameter_from_distance_matrix(D):
    """Diameter of a finite metric space represented by a distance matrix."""
    return int(np.nanmax(D[np.isfinite(D)]))


def fibonacci_cube_metadata(n: int):
    A, vertices = fibonacci_cube_adjacency(n)
    n_components, labels = connected_components(A, directed=False)
    sizes = np.bincount(labels, minlength=n_components)
    return {
        "order": n,
        "vertices": len(vertices),
        "expected_vertices": fibonacci_number(n + 2),
        "edges": A.nnz // 2,
        "n_components": int(n_components),
        "component_sizes": sizes.tolist(),
    }


# Quick sanity checks
for n in range(0, 9):
    meta = fibonacci_cube_metadata(n)
    D, vertices = fibonacci_cube_distance_matrix(n)
    print(
        f"Gamma_{n}: vertices={meta['vertices']} expected={meta['expected_vertices']}, "
        f"edges={meta['edges']}, components={meta['n_components']}, diameter={graph_diameter_from_distance_matrix(D)}"
    )
    assert meta["vertices"] == meta["expected_vertices"]
    assert np.allclose(D, D.T)
    assert np.allclose(np.diag(D), 0)

## 2. Optional graph visualization

Fibonacci cubes are subgraphs of hypercubes. For small `n`, NetworkX spring layouts are useful for intuition.

In [ ]:
def draw_fibonacci_cube(n: int, ax=None, layout="spring", with_labels=True):
    """Draw Gamma_n using NetworkX if available."""
    if not HAS_NETWORKX:
        raise ImportError("networkx is not installed. Run `%pip install networkx` or skip this section.")

    edges, vertices = fibonacci_cube_edges(n)
    G = nx.Graph()
    G.add_nodes_from(range(len(vertices)))
    G.add_edges_from(edges)
    labels = {i: bitstring_to_str(bits) for i, bits in enumerate(vertices)}

    if layout == "spring":
        pos = nx.spring_layout(G, seed=0)
    elif layout == "kamada_kawai":
        pos = nx.kamada_kawai_layout(G)
    elif layout == "spectral":
        pos = nx.spectral_layout(G)
    else:
        raise ValueError("layout must be 'spring', 'kamada_kawai', or 'spectral'")

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))

    nx.draw_networkx_edges(G, pos, ax=ax, width=1.0, alpha=0.7)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=350)
    if with_labels:
        nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8)
    ax.set_title(f"Fibonacci cube $\Gamma_{{{n}}}$")
    ax.axis("off")
    return ax


draw_fibonacci_cube(5);

## 3. Compute Vietoris--Rips persistence with Ripser

The input is the graph shortest-path distance matrix, so we pass `distance_matrix=True` to Ripser.

In [ ]:
def compute_fibonacci_cube_persistence(
    n: int,
    maxdim: int = 3,
    coeff: int = 2,
    thresh=None,
    do_cocycles: bool = False,
):
    """Compute persistent homology of Gamma_n using the graph path metric."""
    D, vertices = fibonacci_cube_distance_matrix(n)
    if thresh is None:
        thresh = graph_diameter_from_distance_matrix(D)

    meta = fibonacci_cube_metadata(n)
    t0 = time.perf_counter()
    result = ripser(
        D,
        distance_matrix=True,
        maxdim=maxdim,
        coeff=coeff,
        thresh=thresh,
        do_cocycles=do_cocycles,
    )
    elapsed = time.perf_counter() - t0

    result.update(meta)
    result["diameter"] = graph_diameter_from_distance_matrix(D)
    result["thresh"] = thresh
    result["maxdim"] = maxdim
    result["coeff"] = coeff
    result["elapsed_seconds"] = elapsed
    result["bitstrings"] = vertices
    return result


example = compute_fibonacci_cube_persistence(8, maxdim=3)
print(f"Computed Gamma_{example['order']} in {example['elapsed_seconds']:.3f}s")
print("vertices:", example["vertices"], "edges:", example["edges"], "diameter:", example["diameter"])
for dim, dgm in enumerate(example["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")
    print(dgm[:10])

## 4. Plot persistence diagrams and barcodes

In [ ]:
def fib_label(result):
    return f"Gamma_{result['order']}"


def plot_persistence_diagram(result, title=None):
    """Plot persistence diagrams from a Ripser result dictionary."""
    if title is None:
        title = f"Persistence diagram for {fib_label(result)}"
    plt.figure(figsize=(6, 6))
    plot_diagrams(result["dgms"], show=False)
    plt.title(title)
    plt.show()


def plot_barcode(result, dims=None, sort_by="birth", title=None, inf_extension=0.25):
    """Plot a barcode for selected homology dimensions."""
    dgms = result["dgms"]
    if dims is None:
        dims = list(range(len(dgms)))

    finite_deaths = []
    for dgm in dgms:
        if len(dgm):
            finite_deaths.extend(dgm[np.isfinite(dgm[:, 1]), 1].tolist())
    max_finite = max(finite_deaths) if finite_deaths else 1.0
    inf_value = max_finite + inf_extension * max(1.0, max_finite)

    fig, ax = plt.subplots(figsize=(9, max(3, 0.25 * sum(len(dgms[d]) for d in dims))))
    y = 0
    yticks = []
    yticklabels = []

    for dim in dims:
        intervals = np.asarray(dgms[dim], dtype=float)
        if len(intervals) == 0:
            continue

        if sort_by == "birth":
            order = np.lexsort((intervals[:, 1], intervals[:, 0]))
        elif sort_by == "persistence":
            deaths_for_sort = intervals[:, 1].copy()
            deaths_for_sort[~np.isfinite(deaths_for_sort)] = inf_value
            order = np.argsort(-(deaths_for_sort - intervals[:, 0]))
        else:
            order = np.arange(len(intervals))

        for idx in order:
            birth, death = intervals[idx]
            death_display = inf_value if not np.isfinite(death) else death
            ax.hlines(y, birth, death_display, linewidth=2)
            if not np.isfinite(death):
                ax.plot(death_display, y, marker=">", markersize=6)
            yticks.append(y)
            yticklabels.append(f"H{dim}")
            y += 1

        ax.axhline(y - 0.5, linewidth=0.5, alpha=0.4)

    ax.set_xlabel("Filtration value")
    ax.set_ylabel("Intervals")
    if len(yticks) <= 60:
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
    else:
        ax.set_yticks([])
    ax.set_title(title or f"Barcode for {fib_label(result)}")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_persistence_diagram(example)
plot_barcode(example, dims=[0, 1, 2, 3], sort_by="persistence")

## 5. Ordinary batch computation through dimension 3

Adjust `ORDERS`, `MAXDIM`, and `COEFF` as desired. Results are cached to disk with `pickle`.

Fibonacci cubes grow with Fibonacci number \(F_{n+2}\) vertices, so the vertex count grows more slowly than \(2^n\) but can still become large for high `n`.

In [ ]:
ORDERS = list(range(1, 15))
MAXDIM = 3
COEFF = 2
THRESH = None
CACHE_FILE = Path(f"fibonacci_cube_persistence_maxdim{MAXDIM}_coeff{COEFF}.pkl")


def batch_compute_fibonacci_cubes(orders, maxdim=3, coeff=2, thresh=None, cache_file=None, force=False):
    """Compute persistence for many Fibonacci cubes and optionally cache results."""
    if cache_file is not None:
        cache_file = Path(cache_file)
        if cache_file.exists() and not force:
            with cache_file.open("rb") as f:
                return pickle.load(f)

    results = {}
    for n in orders:
        print(f"Computing Gamma_{n} ...", end=" ", flush=True)
        try:
            res = compute_fibonacci_cube_persistence(n, maxdim=maxdim, coeff=coeff, thresh=thresh)
            results[n] = res
            print(
                f"done in {res['elapsed_seconds']:.3f}s; vertices={res['vertices']}; "
                f"edges={res['edges']}; diameter={res['diameter']}",
                flush=True,
            )
        except Exception as e:
            print(f"FAILED: {e}", flush=True)
            results[n] = {"order": n, "error": repr(e)}

    if cache_file is not None:
        with cache_file.open("wb") as f:
            pickle.dump(results, f)
    return results


results = batch_compute_fibonacci_cubes(
    ORDERS,
    maxdim=MAXDIM,
    coeff=COEFF,
    thresh=THRESH,
    cache_file=CACHE_FILE,
    force=False,
)

## 6. Summarize persistence across the ordinary batch

In [ ]:
def diagram_stats(dgm):
    """Summary statistics for one persistence diagram."""
    dgm = np.asarray(dgm, dtype=float)
    if len(dgm) == 0:
        return {
            "intervals": 0,
            "finite_intervals": 0,
            "infinite_intervals": 0,
            "max_persistence": 0.0,
            "total_persistence": 0.0,
            "mean_persistence": 0.0,
        }

    finite = np.isfinite(dgm[:, 1])
    pers = dgm[finite, 1] - dgm[finite, 0]
    return {
        "intervals": int(len(dgm)),
        "finite_intervals": int(np.sum(finite)),
        "infinite_intervals": int(np.sum(~finite)),
        "max_persistence": float(np.max(pers)) if len(pers) else 0.0,
        "total_persistence": float(np.sum(pers)) if len(pers) else 0.0,
        "mean_persistence": float(np.mean(pers)) if len(pers) else 0.0,
    }


def summarize_results(results):
    rows = []
    for n, res in results.items():
        if "error" in res:
            rows.append({"order": n, "dimension": None, "error": res["error"]})
            continue
        for dim, dgm in enumerate(res["dgms"]):
            row = {
                "order": n,
                "dimension": dim,
                "vertices": res["vertices"],
                "expected_vertices": res["expected_vertices"],
                "edges": res["edges"],
                "n_components": res["n_components"],
                "diameter": res["diameter"],
                "elapsed_seconds": res["elapsed_seconds"],
                "coeff": res["coeff"],
                "maxdim": res["maxdim"],
            }
            row.update(diagram_stats(dgm))
            rows.append(row)
    return pd.DataFrame(rows)


summary = summarize_results(results)
summary.to_csv("fibonacci_cube_persistence_summary.csv", index=False)
summary.head(16)

## 7. Plot trends across order \(n\)

In [ ]:
def plot_summary_trends(summary, dimensions=None, y="total_persistence"):
    if dimensions is None:
        dimensions = sorted(d for d in summary["dimension"].dropna().unique())

    fig, ax = plt.subplots(figsize=(8, 5))
    for dim in dimensions:
        sub = summary[summary["dimension"] == dim].sort_values("order")
        ax.plot(sub["order"], sub[y], marker="o", label=f"H{int(dim)}")
    ax.set_xlabel("Order n of Fibonacci cube Gamma_n")
    ax.set_ylabel(y.replace("_", " ").title())
    ax.set_title(f"{y.replace('_', ' ').title()} across Fibonacci cubes")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="finite_intervals")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="total_persistence")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="max_persistence")

## 8. Inspect one Fibonacci cube from the ordinary batch

In [ ]:
SELECTED_ORDER = 8
selected = results[SELECTED_ORDER]

print(f"Gamma_{SELECTED_ORDER}: diameter={selected['diameter']}, elapsed={selected['elapsed_seconds']:.3f}s")
print(f"vertices={selected['vertices']}; edges={selected['edges']}; components={selected['n_components']}")
for dim, dgm in enumerate(selected["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")

plot_persistence_diagram(selected)
plot_barcode(selected, dims=list(range(MAXDIM + 1)), sort_by="persistence")

## 9. Optional interactive explorer

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    order_widget = widgets.Dropdown(options=sorted(results.keys()), value=sorted(results.keys())[0], description="order")
    dims_widget = widgets.SelectMultiple(
        options=list(range(MAXDIM + 1)),
        value=tuple(range(MAXDIM + 1)),
        description="dims",
    )
    out = widgets.Output()

    def update(change=None):
        with out:
            clear_output(wait=True)
            n = order_widget.value
            dims = list(dims_widget.value)
            res = results[n]
            if "error" in res:
                print(res["error"])
                return
            print(f"Gamma_{n}: diameter={res['diameter']}, elapsed={res['elapsed_seconds']:.3f}s")
            print(f"vertices={res['vertices']}; edges={res['edges']}; components={res['n_components']}")
            plot_persistence_diagram(res)
            plot_barcode(res, dims=dims, sort_by="persistence")

    order_widget.observe(update, names="value")
    dims_widget.observe(update, names="value")
    display(widgets.HBox([order_widget, dims_widget]), out)
    update()
except Exception as e:
    print("Interactive widgets are unavailable:", e)

## 10. Export ordinary-batch diagrams as data frames

In [ ]:
def diagrams_to_dataframe(result):
    rows = []
    for dim, dgm in enumerate(result["dgms"]):
        for birth, death in dgm:
            rows.append({
                "order": result["order"],
                "vertices": result["vertices"],
                "edges": result["edges"],
                "dimension": dim,
                "birth": float(birth),
                "death": float(death),
                "persistence": float(death - birth) if np.isfinite(death) else np.inf,
            })
    return pd.DataFrame(rows)


all_intervals = pd.concat(
    [diagrams_to_dataframe(res) for res in results.values() if "error" not in res],
    ignore_index=True,
)
all_intervals.to_csv("fibonacci_cube_persistence_intervals.csv", index=False)
all_intervals.head()

## 11. Large-scale experiment: positive-dimensional bars only

This is the analog of the Möbius ladder and knight graph pattern-hunting cells.

It computes Fibonacci cubes \(\Gamma_n\), records all positive-dimensional bars, prints progress, and writes a text file that can be uploaded later for pattern inspection.

This version runs Ripser once per order with a fixed `LARGE_MAXDIM`, because `ripser(..., maxdim=k)` already computes all dimensions up through `k`.

Practical controls:

- `LARGE_MIN_ORDER`, `LARGE_MAX_ORDER`: Fibonacci cube orders to attempt.
- `LARGE_MAXDIM`: largest homology dimension to compute.
- `LARGE_GLOBAL_TIME_LIMIT_SECONDS`: total runtime cap checked between orders.
- `LARGE_PER_ORDER_SOFT_LIMIT_SECONDS`: once a single order takes longer than this, stop the broad run after recording it.
- `LARGE_THRESH`: optional filtration cutoff; `None` means use the diameter.

In [ ]:
# ==========================================================
# Efficient large-scale experiment for Fibonacci cubes
# ==========================================================

LARGE_OUTPUT_TXT = "fibonacci_cube_large_scale_positive_dim_bars.txt"
LARGE_OUTPUT_CSV = "fibonacci_cube_large_scale_positive_dim_bars.csv"
LARGE_SUMMARY_CSV = "fibonacci_cube_large_scale_run_summary.csv"

LARGE_MIN_ORDER = 1
LARGE_MAX_ORDER = 10
LARGE_MAXDIM = 3
LARGE_COEFF = 2
LARGE_THRESH = None
LARGE_GLOBAL_TIME_LIMIT_SECONDS = 30 * 60
LARGE_PER_ORDER_SOFT_LIMIT_SECONDS = 180

large_start_time = time.perf_counter()
bar_rows = []
run_rows = []


def write_large_experiment_outputs():
    """Write current accumulated results to TXT and CSV checkpoint files."""
    bars_df = pd.DataFrame(bar_rows)
    runs_df = pd.DataFrame(run_rows)

    bars_df.to_csv(LARGE_OUTPUT_CSV, index=False)
    runs_df.to_csv(LARGE_SUMMARY_CSV, index=False)

    with open(LARGE_OUTPUT_TXT, "w", encoding="utf-8") as f:
        f.write("# Positive-dimensional persistence bars for Fibonacci cubes\n")
        f.write("# Convention: Gamma_n; path-length metric; Vietoris--Rips persistence via ripser\n")
        f.write("# Columns: order, dimension, bar_index, birth, death, persistence, maxdim_computed, diameter, vertices, edges, coeff, thresh\n")
        for row in bar_rows:
            f.write(
                "order={order}, dim={dimension}, bar_index={bar_index}, "
                "birth={birth}, death={death}, persistence={persistence}, maxdim_computed={maxdim_computed}, "
                "diameter={diameter}, vertices={vertices}, edges={edges}, coeff={coeff}, thresh={thresh}\n".format(**row)
            )


print("Starting Fibonacci-cube large-scale experiment", flush=True)
print("Parameters:", flush=True)
print("  LARGE_MIN_ORDER =", LARGE_MIN_ORDER, flush=True)
print("  LARGE_MAX_ORDER =", LARGE_MAX_ORDER, flush=True)
print("  LARGE_MAXDIM =", LARGE_MAXDIM, flush=True)
print("  LARGE_GLOBAL_TIME_LIMIT_SECONDS =", LARGE_GLOBAL_TIME_LIMIT_SECONDS, flush=True)
print("  LARGE_PER_ORDER_SOFT_LIMIT_SECONDS =", LARGE_PER_ORDER_SOFT_LIMIT_SECONDS, flush=True)

try:
    for n in range(LARGE_MIN_ORDER, LARGE_MAX_ORDER + 1):
        elapsed_global = time.perf_counter() - large_start_time
        if elapsed_global >= LARGE_GLOBAL_TIME_LIMIT_SECONDS:
            print("Global time limit reached before starting next order.", flush=True)
            break

        print("", flush=True)
        print("=" * 72, flush=True)
        print("Starting Gamma_{} at global elapsed {:.1f}s".format(n, elapsed_global), flush=True)
        print("=" * 72, flush=True)

        try:
            D, vertices = fibonacci_cube_distance_matrix(n)
            meta = fibonacci_cube_metadata(n)
            diameter = graph_diameter_from_distance_matrix(D)
            thresh = diameter if LARGE_THRESH is None else LARGE_THRESH

            print(
                "Graph built: vertices={}, edges={}, components={}, diameter={}, thresh={}".format(
                    meta["vertices"], meta["edges"], meta["n_components"], diameter, thresh
                ),
                flush=True,
            )
        except Exception as e:
            print("Failed to build Gamma_{}: {}".format(n, repr(e)), flush=True)
            run_rows.append({
                "order": n,
                "status": "graph_build_failed",
                "maxdim_computed": None,
                "seconds": 0.0,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            continue

        print("Running ripser for Gamma_{}, maxdim={} ...".format(n, LARGE_MAXDIM), end=" ", flush=True)
        t0 = time.perf_counter()

        try:
            result = ripser(
                D,
                distance_matrix=True,
                maxdim=LARGE_MAXDIM,
                coeff=LARGE_COEFF,
                thresh=thresh,
            )
            seconds = time.perf_counter() - t0
            interval_counts = [len(dgm) for dgm in result["dgms"]]
            print("done in {:.2f}s; interval counts={}".format(seconds, interval_counts), flush=True)

            positive_bar_count = 0
            for dim in range(1, len(result["dgms"])):
                dgm = result["dgms"][dim]
                for bar_index, pair in enumerate(dgm):
                    birth = float(pair[0])
                    death = float(pair[1])
                    persistence = float(death - birth) if np.isfinite(death) else np.inf
                    bar_rows.append({
                        "order": n,
                        "dimension": dim,
                        "bar_index": bar_index,
                        "birth": birth,
                        "death": death,
                        "persistence": persistence,
                        "maxdim_computed": LARGE_MAXDIM,
                        "diameter": diameter,
                        "vertices": meta["vertices"],
                        "edges": meta["edges"],
                        "coeff": LARGE_COEFF,
                        "thresh": thresh,
                    })
                    positive_bar_count += 1

            run_rows.append({
                "order": n,
                "status": "success",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": positive_bar_count,
                "message": "",
            })

            print("Recorded {} positive-dimensional bars for Gamma_{}.".format(positive_bar_count, n), flush=True)
            write_large_experiment_outputs()

            if seconds >= LARGE_PER_ORDER_SOFT_LIMIT_SECONDS:
                print("This order exceeded the per-order soft limit. Stopping broad run.", flush=True)
                break

        except Exception as e:
            seconds = time.perf_counter() - t0
            print("failed after {:.2f}s: {}".format(seconds, repr(e)), flush=True)
            run_rows.append({
                "order": n,
                "status": "failed",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            break

except KeyboardInterrupt:
    print("", flush=True)
    print("KeyboardInterrupt received. Writing partial results.", flush=True)
    write_large_experiment_outputs()

print("", flush=True)
print("Fibonacci-cube large-scale experiment finished or interrupted safely.", flush=True)
print("Total positive-dimensional bars recorded:", len(bar_rows), flush=True)
print("Text output:", LARGE_OUTPUT_TXT, flush=True)
print("CSV output:", LARGE_OUTPUT_CSV, flush=True)
print("Run summary:", LARGE_SUMMARY_CSV, flush=True)

pd.DataFrame(run_rows).tail()

## 12. Optional: compare with hypercubes

Since Fibonacci cubes are induced subgraphs of hypercubes, it can be useful to compare them with full hypercubes of the same bit-length. This cell is intentionally small-scale.

In [ ]:
def hypercube_edges(n: int):
    """Edges of the n-dimensional hypercube Q_n."""
    if n < 0:
        raise ValueError("n must be nonnegative")
    vertices = list(range(2**n))
    edges = set()
    for x in vertices:
        for j in range(n):
            y = x ^ (1 << j)
            edges.add(tuple(sorted((x, y))))
    return sorted(edges)


def hypercube_distance_matrix(n: int):
    """Shortest-path distance matrix of Q_n."""
    edges = hypercube_edges(n)
    m = 2**n
    rows, cols = [], []
    for u, v in edges:
        rows.extend([u, v])
        cols.extend([v, u])
    A = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(m, m))
    D = shortest_path(A, directed=False, unweighted=True)
    return np.asarray(D, dtype=float)

# Example comparison for small n
COMPARE_N = 5
Dq = hypercube_distance_matrix(COMPARE_N)
q_result = ripser(Dq, distance_matrix=True, maxdim=3, thresh=graph_diameter_from_distance_matrix(Dq))
print(f"Q_{COMPARE_N}: vertices={2**COMPARE_N}, diameter={graph_diameter_from_distance_matrix(Dq)}")
for dim, dgm in enumerate(q_result["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")

## 13. Ideas for further experiments

- Compare \(\Gamma_n\) with \(Q_n\), the full hypercube.
- Try larger `LARGE_MAXDIM` for small and moderate `n`.
- Compare coefficient fields, e.g. `coeff=2`, `coeff=3`, and `coeff=5`.
- Test for clean barcode formulas by uploading `fibonacci_cube_large_scale_positive_dim_bars.txt` after a run.
- Investigate whether all low-dimensional bars are concentrated at small integer filtration values, as in the knight graphs, or whether Fibonacci cubes exhibit longer global bars like the Möbius ladders.